# Explore Chroma Vector Store
Inspect chunks, metadata, and run similarity searches against the local DB.

In [6]:
import sys
sys.path.insert(0, '..')  # make app/ importable

import chromadb

client = chromadb.PersistentClient(path='../chroma_db')
col = client.get_collection('bayer-rag')

print('Collections:', client.list_collections())
print('Total chunks:', col.count())

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Collections: ['bayer-rag']
Total chunks: 15


## Peek at first 5 chunks

In [2]:
results = col.peek(5)

for i, (doc_id, doc, meta) in enumerate(zip(results['ids'], results['documents'], results['metadatas'])):
    print(f'--- Chunk {i} ---')
    print(f'ID:     {doc_id}')
    print(f'Source: {meta.get("source", "unknown")}')
    print(f'Text:   {doc[:300]}')
    print()

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


--- Chunk 0 ---
ID:     731d503f98571580f9122ca5ba8835f9
Source: documents/claims_guidelines.pdf
Text:   Consumer Health Claims & Evidence Guidelines (Sample)
Owner: Medical, Legal, Regulatory (MLR)
Applies to: AU/NZ (example), Digital + Retail + HCP-lite content
Version: 1.3
Last Updated: 2026-01-10
1. Purpose
Define what types of claims are permitted for consumer health marketing content and what
evi

--- Chunk 1 ---
ID:     c8a8061680f137b8b094bb07524faa3f
Source: documents/claims_guidelines.pdf
Text:   B. Benefit Claims (Structure or Function)
- “Supports immune system function”
Evidence: accepted ingredients, dose rationale, substantiation dossier.
C. Symptom Relief Claims
- “Relieves headache pain”
Evidence: clinical trial data, monograph alignment, approved label.
D. Comparative Claims
- “Faste

--- Chunk 2 ---
ID:     c29bc9cf6f4f30b4ccf43d0a236477e0
Source: documents/claims_guidelines.pdf
Text:   Default: Not permitted unless explicitly approved with strong evidence (rare).
3.

## All chunks grouped by source file

In [3]:
from collections import Counter

all_meta = col.get(include=['metadatas'])['metadatas']
sources = Counter(m.get('source', 'unknown') for m in all_meta)

print(f'{'Source':<60} Chunks')
print('-' * 70)
for source, count in sorted(sources.items()):
    print(f'{source:<60} {count}')

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Source                                                       Chunks
----------------------------------------------------------------------
documents/claims_guidelines.pdf                              6
documents/marketing_policy.md                                6
documents/product_facts.txt                                  3


## Similarity search (no LLM — raw vector retrieval)

In [4]:
from dotenv import load_dotenv
load_dotenv('../.env')

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import os

embeddings = OpenAIEmbeddings(
    model='text-embedding-3-small',
    api_key=os.environ['OPENAI_API_KEY'],
)

vector_store = Chroma(
    client=client,
    collection_name='bayer-rag',
    embedding_function=embeddings,
)

QUERY = 'What are the active ingredients?'

docs = vector_store.similarity_search_with_score(QUERY, k=4)

for doc, score in docs:
    print(f'Score (distance): {score:.4f}')
    print(f'Source: {doc.metadata.get("source")}')
    print(f'Text:   {doc.page_content[:300]}')
    print()

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Score (distance): 1.1277
Source: documents/claims_guidelines.pdf
Text:   - Benefit: ingredient monograph + dose rationale
- Relief: clinical evidence + label alignment
- Comparative: head-to-head study + approved comparator framing
5. Required Disclaimers (examples)
- “Always read the label and follow the directions for use.”
- “If symptoms persist, talk to your healthca

Score (distance): 1.1895
Source: documents/product_facts.txt
Text:   Document: Product Factsheet
Brand: ExampleBrand
SKU: EXB-VC-1000-20
Region: AU/NZ (example)
Version: 2.0
Last Updated: 2026-02-02

1. Product Name
ExampleBrand Vitamin C 1000 mg Effervescent Tablets (20s)

2. Indications (Label-aligned)
- Vitamin C supplementation
- Helps maintain immune system func

Score (distance): 1.2290
Source: documents/product_facts.txt
Text:   4. Key Product Features
- Effervescent format
- Orange flavour
- Gluten-free (example)
- No artificial colours (example)

5. Directions for Use
Adults: Dissolve 1 tablet in a glass of w

## Peek at raw vector dimensions

In [5]:
result = col.get(include=['embeddings'], limit=1)
vector = result['embeddings'][0]

print(f'Vector dimensions: {len(vector)}')
print(f'First 10 values:   {[round(v, 6) for v in vector[:10]]}')

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Vector dimensions: 1536
First 10 values:   [np.float64(0.000625), np.float64(0.003988), np.float64(0.010835), np.float64(0.048238), np.float64(-0.017657), np.float64(-0.012564), np.float64(-0.009857), np.float64(-0.003401), np.float64(-0.003057), np.float64(0.031376)]
